# Notebook 06 – Model Evaluation

Dieses Notebook führt die finale Evaluation der traineirten RL-Agenten durch.

Dafür wird sich an die folgenden Regeln gehalten:
- Finale Evaluation wird auf 50 Episoden pro Run gesetzt
- Wichtig: Alte 200-Episoden-Evaluations-Runs wurden manuell archiviert und nicht mehr in die finale Auswertung geladen (eine genauere Erklärung dazu findet sich weiter unten)
- A2C und DQN werden in Env1 deterministisch und stochastisch evaluiert (Begründung erfolgt weiter unten)
- PPO wird separat über Unity ML-Agents Inference behandelt
- Transfer wird nur mit den besten Modellen aus der Evaluation des Env1 durchgeführt (genauer Begründung erfolgt ebenfalls weiter unten)

---

### 1. Imports und Projektpfade

Diese Zelle lädt die benötigten Bibliotheken und setzt die zentralen Projektpfade für Modelle, Logs und Evaluationsergebnisse.

In [ ]:
from __future__ import annotations

import sys
import os
import json
import time
import random
import warnings
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import A2C, DQN

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper

warnings.filterwarnings("ignore")

# Projektwurzel wie in Notebook 05 setzen
PROJECT_ROOT_NOTEBOOK = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT_NOTEBOOK))

from lib_py.paths import (
    PROJECT_ROOT,
    PROJECT_NAME,
    MAZE_AGENT_DIR,
    TRAINING_DIR,
    MODEL_DIR,
    LOG_DIR,
    ensure_project_dirs,
    show_project_path,
)

# Working Directory auf Projektwurzel setzen
os.chdir(PROJECT_ROOT)

# Zentrale Projektordner
ensure_project_dirs()

CONFIG_DIR = TRAINING_DIR / "configs"
TUNING_DIR = TRAINING_DIR / "tuning"
FINAL_MODEL_DIR = MODEL_DIR / "final"
FINAL_LOG_DIR = LOG_DIR / "final"
RUN_SUMMARY_DIR = TRAINING_DIR / "run_summaries"
RECORDING_DIR = TRAINING_DIR / "recordings"

EVALUATION_DIR = TRAINING_DIR / "evaluation_results"
EVALUATION_RAW_DIR = EVALUATION_DIR / "raw"
EVALUATION_TABLE_DIR = EVALUATION_DIR / "tables"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TUNING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
RECORDING_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_RAW_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Imports funktionieren.")
print("Project Name:", PROJECT_NAME)
print("Project Root:", show_project_path(PROJECT_ROOT))
print("Current Working Directory:", show_project_path(Path.cwd()))
print("Final Model Dir:", show_project_path(FINAL_MODEL_DIR))
print("Final Log Dir:", show_project_path(FINAL_LOG_DIR))
print("Run Summary Dir:", show_project_path(RUN_SUMMARY_DIR))
print("Evaluation Raw Dir:", show_project_path(EVALUATION_RAW_DIR))
print("Evaluation Table Dir:", show_project_path(EVALUATION_TABLE_DIR))

---

### 2. Evaluation-Konfiguration

Diese Zelle legt die zentralen Evaluationseinstellungen fest. Geplant sind 200 Episoden pro Modell und Umgebung.

In [ ]:
EVAL_EPISODES = 50
EVAL_MAX_STEPS_PER_EPISODE = 5000

TRAIN_ENV_NAME = "Env1"

EVAL_ENVIRONMENTS = [
    "Env1",
    "Env2",
    "Env3",
]

print("Evaluation Episodes:", EVAL_EPISODES)
print("Max Steps pro Episode:", EVAL_MAX_STEPS_PER_EPISODE)
print("Trainingsumgebung:", TRAIN_ENV_NAME)
print("Evaluation Environments:", EVAL_ENVIRONMENTS)


### 2.5 NumPy-Kompatibilität für gespeicherte SB3-Modelle

Diese Zelle verhindert Ladefehler bei SB3-Modellen, die mit einer anderen Numpy-Version gespeichert wurden

In [ ]:
try:
    import numpy.core as numpy_core
    import numpy.core.multiarray as numpy_core_multiarray
    import numpy.core.numeric as numpy_core_numeric
    import numpy.core._multiarray_umath as numpy_core_multiarray_umath

    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", numpy_core_multiarray)
    sys.modules.setdefault("numpy._core.numeric", numpy_core_numeric)
    sys.modules.setdefault("numpy._core._multiarray_umath", numpy_core_multiarray_umath)

    print("NumPy compatibility aliases gesetzt.")
    print("NumPy Version:", np.__version__)

except Exception as exc:
    print("NumPy compatibility aliases konnten nicht gesetzt werden:")
    print(type(exc).__name__, exc)

### 3. Globale Seeds setzen

Diese Funktion setzt Python-, NumPy- und Random-Seeds, damit die Evaluation reproduzierbarer wird

In [ ]:
def set_global_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    try:
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    print(f"Seeds gesetzt auf: {seed}")

---

### 4. Unity-Environment erstellen

Diese Funktion verbindet Python mit der aktuell geöffneten Unity-Szene und wrapped sie als Gym-Environment.  

Für A2C und DQN gilt:
- Unity-Szene öffnen
- MazeAgent Behavior Type auf `Default`
- kein ONNX-Modell eintragen
- Python-Zelle starten
- anschließend in Unity Play drücken

In [ ]:
def make_unity_env(seed: int, worker_id: int = 0):
    set_global_seeds(seed)

    print("Starte Verbindung zu Unity. Jetzt in Unity Play drücken...")

    unity_env = UnityEnvironment(
        file_name=None,
        seed=seed,
        worker_id=worker_id,
        no_graphics=False,
    )

    env = UnityToGymWrapper(
        unity_env,
        uint8_visual=False,
        flatten_branched=True,
        allow_multiple_obs=False,
    )

    print("Unity Environment verbunden.")
    print("Observation Space:", env.observation_space)
    print("Action Space:", env.action_space)

    return env

---

### 5. Hilfsfunktionen für SB3-Umgebungen

Diese Funktionen vereinheitlichen Reset, Step und Done-Handling, weil Unity-/SB3-Wrapper teilweise leicht unterschiedliche Rückgabeformate verwenden.

In [ ]:
def reset_env(env):
    reset_result = env.reset()

    if isinstance(reset_result, tuple) and len(reset_result) == 2:
        obs, info = reset_result
        return obs

    return reset_result


def step_env(env, action):
    step_result = env.step(action)

    if isinstance(step_result, tuple) and len(step_result) == 5:
        obs, reward, terminated, truncated, info = step_result
        done = terminated or truncated
        return obs, reward, done, info

    if isinstance(step_result, tuple) and len(step_result) == 4:
        obs, reward, done, info = step_result
        return obs, reward, done, info

    raise ValueError(f"Unbekanntes env.step() Rückgabeformat: {type(step_result)}")


def to_scalar(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.array(value).flatten()

        if len(arr) == 0:
            return None

        return arr[0].item()

    return value


def done_to_bool(done) -> bool:
    return bool(to_scalar(done))


def reward_to_float(reward) -> float:
    return float(to_scalar(reward))


def info_to_dict(info) -> dict:
    if isinstance(info, list) and len(info) > 0:
        if isinstance(info[0], dict):
            return info[0]
        return {"info": str(info[0])}

    if isinstance(info, dict):
        return info

    return {"info": str(info)}

---

### 6. Run-Summaries laden

Diese Zelle lädt die bereinigten Run-Summary-Dateien. Für DQN wird die bereinigte DQN-v1-Datei verwendet. 

In [ ]:
def load_csv_if_exists(path: Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        print(f"{label} geladen:", show_project_path(path), f"({len(df)} Zeilen)")
        return df

    print(f"{label} nicht gefunden:", show_project_path(path))
    return pd.DataFrame()


a2c_runs_df = load_csv_if_exists(
    RUN_SUMMARY_DIR / "a2c_final_training_runs.csv",
    "A2C Runs"
)

dqn_runs_df = load_csv_if_exists(
    RUN_SUMMARY_DIR / "dqn_v1_final_evaluation_runs.csv",
    "DQN v1 Runs"
)

display(a2c_runs_df.head())
display(dqn_runs_df.head())

---

### 7. Modellpfade bereinigen und prüfen

Diese Funktionen bestimmen den Pfad zu einem gespeicherten SB3-Modell. Falls der gespeicherte Pfad in der Summary nicht direkt passt, wird zusätzlich im finalen Modellordner gesucht (sollte egtl nicht nötig sein)

In [ ]:
def normalize_model_path(model_path_value) -> Path | None:
    if pd.isna(model_path_value):
        return None

    model_path = Path(str(model_path_value))

    if not model_path.is_absolute():
        model_path = PROJECT_ROOT / model_path

    return model_path


def candidate_model_paths_from_run_ids(row: pd.Series) -> list[Path]:
    candidates = []

    seed = int(row["seed"])
    algorithm = row["algorithm"]

    possible_run_ids = []

    for col in ["run_id", "original_run_id"]:
        if col in row.index and not pd.isna(row[col]):
            possible_run_ids.append(str(row[col]))

    if algorithm == "A2C":
        possible_run_ids.extend([
            f"A2C_Env1_Seed{seed}",
            f"A2C_Seed{seed}",
        ])

    if algorithm == "DQN":
        possible_run_ids.extend([
            f"DQN_Env1_Seed{seed}",
            f"DQN_Env1_Seed{seed}_v2",
            f"DQN_v1_Seed{seed}",
        ])

    possible_run_ids = list(dict.fromkeys(possible_run_ids))

    for run_id in possible_run_ids:
        candidates.extend([
            FINAL_MODEL_DIR / run_id / "final_model.zip",
            FINAL_MODEL_DIR / run_id / "final_model",
        ])

    return candidates


def find_sb3_model_path(row: pd.Series) -> Path | None:
    if "model_path" in row.index:
        candidate = normalize_model_path(row["model_path"])

        if candidate is not None:
            if candidate.exists():
                return candidate

            if candidate.with_suffix(".zip").exists():
                return candidate.with_suffix(".zip")

    for candidate in candidate_model_paths_from_run_ids(row):
        if candidate.exists():
            return candidate

    return None


def add_clean_labels(df: pd.DataFrame, algorithm: str) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    out = df.copy()
    out["algorithm"] = algorithm

    if "original_run_id" not in out.columns:
        out["original_run_id"] = out["run_id"]

    if "final_analysis_label" not in out.columns:
        out["final_analysis_label"] = np.nan

    for idx, row in out.iterrows():
        seed = int(row["seed"])

        if pd.isna(row.get("final_analysis_label")):
            if algorithm == "A2C":
                out.loc[idx, "final_analysis_label"] = f"A2C_Seed{seed}"
            elif algorithm == "DQN":
                out.loc[idx, "final_analysis_label"] = f"DQN_v1_Seed{seed}"

    out["analysis_run_id"] = out["final_analysis_label"]

    out["resolved_model_path"] = out.apply(find_sb3_model_path, axis=1)
    out["model_exists"] = out["resolved_model_path"].apply(
        lambda path: path is not None and Path(path).exists()
    )

    out["resolved_model_path_display"] = out["resolved_model_path"].apply(
        lambda path: show_project_path(Path(path)) if path is not None else None
    )

    return out


a2c_models_df = add_clean_labels(a2c_runs_df, "A2C")
dqn_models_df = add_clean_labels(dqn_runs_df, "DQN")

sb3_models_df = pd.concat(
    [a2c_models_df, dqn_models_df],
    ignore_index=True,
    sort=False
)

display(sb3_models_df[[
    "analysis_run_id",
    "original_run_id",
    "algorithm",
    "seed",
    "resolved_model_path_display",
    "model_exists"
]])

### 7.5 Offizielle SB3-Modelle auswählen

Diese Zelle filtert auf Modelle, die wirklich existieren und evaluiert werden können. Falls ein Modellpfad fehlt, wird es angezeigt und nicht automatisch evaluiert

In [ ]:
sb3_eval_models_df = sb3_models_df[sb3_models_df["model_exists"] == True].copy()
missing_models_df = sb3_models_df[sb3_models_df["model_exists"] != True].copy()

print("Gefundene SB3-Modelle:", len(sb3_eval_models_df))

display(sb3_eval_models_df[[
    "analysis_run_id",
    "algorithm",
    "seed",
    "resolved_model_path_display"
]])

if len(missing_models_df) > 0:
    print("Fehlende Modelle:")

    display(missing_models_df[[
        "analysis_run_id",
        "algorithm",
        "seed",
        "original_run_id",
        "resolved_model_path_display"
    ]])

---

### 8. Einzelne SB3-Modelle evaluieren

Bei dierser Funktion wird ein A2C- oder DQN-Modell in der aktuell geöffneten Unity-Szene evaluiert

Wichtige Änderungen:
- Run-ID enthält jetzt den Evaluationsmodus (deterministisch/stochastisch)
- Pro Run werden standardmäßig 50 Episoden evaluiert
- Action Counts werden gespeichert, um bestimmtes Verhalten wie z.b. frozen-agent- oder no-op-verhalten analysieren zu können
- beim Laden des Modells werden `custom_objects` verwendet, um Gym-/Gymnasium-/Numpy-Kompatibilitätsprobleme zu umgehen
- die info spalte wird leer gespeichert, damit keine unlesbaren Speicherobjekte in der CSV landen

In [ ]:
def evaluate_single_sb3_model(
    model_row: pd.Series,
    eval_environment: str,
    n_episodes: int = EVAL_EPISODES,
    max_steps_per_episode: int = EVAL_MAX_STEPS_PER_EPISODE,
    deterministic: bool = True,
) -> pd.DataFrame:
    algorithm = model_row["algorithm"]
    seed = int(model_row["seed"])
    analysis_run_id = model_row["analysis_run_id"]
    model_path = Path(model_row["resolved_model_path"])

    if algorithm == "A2C":
        model_class = A2C
    elif algorithm == "DQN":
        model_class = DQN
    else:
        raise ValueError(f"Nicht unterstützter SB3-Algorithmus: {algorithm}")

    eval_mode_suffix = "deterministic" if deterministic else "stochastic"
    eval_run_id = f"{analysis_run_id}_eval_{eval_environment}_{eval_mode_suffix}"

    print("=" * 80)
    print("Evaluation Run:", eval_run_id)
    print("Algorithmus:", algorithm)
    print("Seed:", seed)
    print("Modell:", show_project_path(model_path))
    print("Evaluation Environment:", eval_environment)
    print("Episoden:", n_episodes)
    print("Deterministic:", deterministic)
    print("=" * 80)
    print("Bitte jetzt in Unity die passende Szene öffnen und Play drücken.")
    print("Wenn Unity verbunden ist, startet die Evaluation automatisch.")

    set_global_seeds(seed)

    env = None
    rows = []

    try:
        env = make_unity_env(seed)

        custom_objects = {
            "observation_space": env.observation_space,
            "action_space": env.action_space,
        }

        model = model_class.load(
            str(model_path),
            env=env,
            custom_objects=custom_objects,
            force_reset=True,
        )

        for episode_idx in range(1, n_episodes + 1):
            obs = reset_env(env)
            done = False

            episode_reward = 0.0
            episode_steps = 0
            action_counts = {}

            while not done and episode_steps < max_steps_per_episode:
                action, _ = model.predict(obs, deterministic=deterministic)

                action_scalar = int(np.array(action).flatten()[0])
                action_counts[action_scalar] = action_counts.get(action_scalar, 0) + 1

                obs, reward, done, info = step_env(env, action)

                episode_reward += reward_to_float(reward)
                episode_steps += 1
                done = done_to_bool(done)

            timeout_by_python_limit = episode_steps >= max_steps_per_episode and not done
            most_common_action = max(action_counts, key=action_counts.get) if action_counts else None

            rows.append({
                "eval_run_id": eval_run_id,
                "analysis_run_id": analysis_run_id,
                "original_run_id": model_row.get("original_run_id", np.nan),
                "algorithm": algorithm,
                "train_environment": TRAIN_ENV_NAME,
                "eval_environment": eval_environment,
                "seed": seed,
                "episode": episode_idx,
                "episode_reward": episode_reward,
                "episode_steps": episode_steps,
                "done": done,
                "timeout_by_python_limit": timeout_by_python_limit,
                "deterministic": deterministic,
                "eval_mode": eval_mode_suffix,
                "most_common_action": most_common_action,
                "action_counts": str(action_counts),
                "model_path": show_project_path(model_path),
                "info": "",
                "evaluated_at": pd.Timestamp.now().isoformat(timespec="seconds"),
            })

            if episode_idx % 10 == 0:
                recent_rewards = [row["episode_reward"] for row in rows[-10:]]
                recent_steps = [row["episode_steps"] for row in rows[-10:]]

                print(
                    f"{eval_run_id}: Episode {episode_idx}/{n_episodes} | "
                    f"Last10 Reward: {np.mean(recent_rewards):.3f} | "
                    f"Last10 Steps: {np.mean(recent_steps):.1f}"
                )

        eval_df = pd.DataFrame(rows)

        output_path = EVALUATION_RAW_DIR / f"{eval_run_id}.csv"
        eval_df.to_csv(output_path, index=False, encoding="utf-8")

        print("Evaluation gespeichert:")
        print(show_project_path(output_path))

        return eval_df

    finally:
        if env is not None:
            env.close()
            print(f"{eval_run_id}: Unity Environment geschlossen.")

---

### 9. Modell gezielt auswählen und evaluieren

Diese Funktion sucht aus der Modellübersicht ein bestimmtes Modell anhand von Algorithmus und Seed heraus und evaluiert es in der aktuell geöffneten Unity-Umgebung.

Dadurch können Evaluationen einzeln gestartet, beobachtet und aufgezeichnet werden was ziemlich wichtig für die Dokumentation des Projekts ist

In [ ]:
def get_model_row(
    algorithm: str,
    seed: int,
    models_df: pd.DataFrame = sb3_eval_models_df,
) -> pd.Series:
    matches = models_df[
        (models_df["algorithm"] == algorithm) &
        (models_df["seed"].astype(int) == int(seed))
    ]

    if len(matches) == 0:
        raise ValueError(f"Kein Modell gefunden für algorithm={algorithm}, seed={seed}")

    if len(matches) > 1:
        print("Warnung: Mehrere passende Modelle gefunden. Es wird das erste verwendet.")
        display(matches)

    return matches.iloc[0]


def evaluate_selected_sb3_model(
    algorithm: str,
    seed: int,
    eval_environment: str,
    n_episodes: int = EVAL_EPISODES,
    deterministic: bool = DETERMINISTIC_EVAL,
) -> pd.DataFrame:
    model_row = get_model_row(
        algorithm=algorithm,
        seed=seed,
        models_df=sb3_eval_models_df,
    )

    return evaluate_single_sb3_model(
        model_row=model_row,
        eval_environment=eval_environment,
        n_episodes=n_episodes,
        deterministic=deterministic,
    )

---

### 10. Modell gezielt auswählen und evaluieren

Diese Funktionen wählen anhand von Algorithmus und Seed das passende SB3-Modell aus und starten die Evaluation.

In [ ]:
def get_model_row(
    algorithm: str,
    seed: int,
    models_df: pd.DataFrame = sb3_eval_models_df,
) -> pd.Series:
    matches = models_df[
        (models_df["algorithm"] == algorithm) &
        (models_df["seed"].astype(int) == int(seed))
    ]

    if len(matches) == 0:
        raise ValueError(f"Kein Modell gefunden für algorithm={algorithm}, seed={seed}")

    if len(matches) > 1:
        print("Warnung: Mehrere passende Modelle gefunden. Es wird das erste verwendet.")
        display(matches)

    return matches.iloc[0]


def evaluate_selected_sb3_model(
    algorithm: str,
    seed: int,
    eval_environment: str,
    n_episodes: int = EVAL_EPISODES,
    deterministic: bool = True,
) -> pd.DataFrame:
    model_row = get_model_row(
        algorithm=algorithm,
        seed=seed,
        models_df=sb3_eval_models_df,
    )

    return evaluate_single_sb3_model(
        model_row=model_row,
        eval_environment=eval_environment,
        n_episodes=n_episodes,
        deterministic=deterministic,
    )

---

### 11. A2C Env1 - deterministisch

A2C wird zunächst deterministisch in Env1 evaluiert.  
Die deterministische Evaluation prüft, wie robust die finale Policy ist, wenn immer die bevorzugte Aktion gewählt wird.

Vor jedem Run in Unity:
- Env1 öffnen
- Behavior Type: Default
- Model: None
- EpisodeLogger Run ID passend setzen
- Max Episodes: 50

#### 11.1 A2C Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle deterministisch in Env1 mit Seed 1

In [ ]:
eval_a2c_seed1_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=1,
    eval_environment="Env1",
    deterministic=True,
)

#### 11.2 A2C Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle deterministisch in Env1 mit Seed 27

In [ ]:
eval_a2c_seed27_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=27,
    eval_environment="Env1",
    deterministic=True,
)

#### 11.3 A2C Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle deterministisch in Env1 mit Seed 42.

In [ ]:
eval_a2c_seed42_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=42,
    eval_environment="Env1",
    deterministic=True,
)

---

### 12. A2C Env1 – stochastic

A2C wird zusätzlich stochastisch evaluiert.  
Dabei wird nicht immer die deterministisch bevorzugte Aktion genutzt, sondern aus der gelernten Policy-Verteilung gesampelt.

#### 12.1 A2C Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle stochastisch in Env1 mit Seed 1

In [ ]:
eval_a2c_seed1_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=1,
    eval_environment="Env1",
    deterministic=False,
)

#### 12.2 A2C Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle stochastisch in Env1 mit Seed 27

In [ ]:
eval_a2c_seed27_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=27,
    eval_environment="Env1",
    deterministic=False,
)

#### 12.3 A2C Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle stochastisch in Env1 mit Seed 42.

In [ ]:
eval_a2c_seed42_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=42,
    eval_environment="Env1",
    deterministic=False,
)

---

### 13. DQN Env1 - deterministisch

DQN wird zunächst deterministisch in Env1 evaluiert.  
Diese Evaluation dient vor allem als Robustheitsdiagnose der greedy Policy. Das ist besonders relevant, da bei DQN Seed 27 bereits in ersten Testläufen ein Frozen-Agent-Verhalten in Zielnähe beobachtet wurde.

*Hinweis: Das beobachtete Frozen-Agent-Verhalten ist der zentrale Grund dafür, zusätzlich zur deterministischen Evaluation auch eine stochastische Evaluation durchzuführen. In der deterministischen Evaluation wählt der Agent stets die greedy Action. Wenn diese Aktion in Zielnähe zu einem No-Op oder einer ineffektiven Bewegung führt, kann der Agent reproduzierbar in einer suboptimalen Situation einfrieren. Erste Testläufe zeigten genau dieses Muster.Während deterministische Durchläufe dazu führten, dass der Agent in direkter Zielnähe stehen blieb, konnten in stochastischen Durchläufen Goal-Events erreicht werden. Deshalb werden DQN-Modelle in beiden Modi evaluiert.*

*Hinweis 2: Aus Gründen der wissenschaftlichen Nachvollziehbarkeit und Konsistenz wird auch A2C in beiden Modi (deterministisch und stochastisch) evaluiert. Diese Entscheidung wurde erst nach der Beobachtung und Analyse der DQN-Ergebnisse getroffen, auch wenn A2C in der Notebook-Struktur vor DQN ausgewertet wird.*

#### 13.1 DQN Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 1

In [ ]:
eval_dqn_seed1_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=1,
    eval_environment="Env1",
    deterministic=True,
)

#### 13.2 DQN Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 27

In [ ]:
eval_dqn_seed27_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=27,
    eval_environment="Env1",
    deterministic=True,
)

#### 13.3 DQN Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 42

In [ ]:
eval_dqn_seed42_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=42,
    eval_environment="Env1",
    deterministic=True,
)

#### 13.4 DQN Seed 72 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 72

In [ ]:
eval_dqn_seed72_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=72,
    eval_environment="Env1",
    deterministic=True,
)

#### 13.5 DQN Seed 100 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 100

In [ ]:
eval_dqn_seed100_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=100,
    eval_environment="Env1",
    deterministic=True,
)

---

### 14. DQN Env1 – stochastic

Wie im Abschnitt 13. schon erklärt wird DQN zusätzlich stochastisch evaluiert.  
Diese Evaluation prüft, ob die Modelle grundsätzlich zielrelevantes Verhalten gelernt haben, auch wenn die rein greedy Policy nicht robust ist.

Die Ergebnisse werden getrennt von der deterministischen Evaluation berichtet.

#### 14.1 DQN Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 1

In [ ]:
eval_dqn_seed1_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=1,
    eval_environment="Env1",
    deterministic=False,
)

#### 14.2 DQN Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 27

In [ ]:
eval_dqn_seed27_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=27,
    eval_environment="Env1",
    deterministic=False,
)

#### 14.3 DQN Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 42

In [ ]:
eval_dqn_seed42_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=42,
    eval_environment="Env1",
    deterministic=False,
)

#### 14.4 DQN Seed 72 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 72

In [ ]:
eval_dqn_seed72_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=72,
    eval_environment="Env1",
    deterministic=False,
)

#### 14.5 DQN Seed 100 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 100

In [ ]:
eval_dqn_seed100_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=100,
    eval_environment="Env1",
    deterministic=False,
)

---

### 15. PPO Evaluation über Unity ML-Agents Inference

PPO wird separat evaluiert, da PPO über Unity ML-Agents trainiert wurde und nicht als Stable-Baselines3-Modell geladen wird.

Für PPO gilt:
- ONNX-Modell in Unity beim MazeAgent eintragen
- Behavior Type: Inference Only
- passende Szene öffnen
- EpisodeLogger aktivieren
- Max Episodes: 50

PPO wird nicht in dieselbe deterministic/stochastic-Struktur wie A2C und DQN gepresst.  
Die Ergebnisse werden separat als native Unity-ML-Agents-Inference-Evaluation dokumentiert.

In [ ]:
ppo_eval_plan_df = pd.DataFrame([
    {"algorithm": "PPO", "seed": 1, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed1_eval_Env1"},
    {"algorithm": "PPO", "seed": 27, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed27_eval_Env1"},
    {"algorithm": "PPO", "seed": 42, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed42_eval_Env1"},
    {"algorithm": "PPO", "seed": 72, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed72_eval_Env1"},
    {"algorithm": "PPO", "seed": 100, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed100_eval_Env1"},
])

display(ppo_eval_plan_df)

---

### 16. Finale Evaluation-CSV-Dateien laden

Diese Funktion lädt alle finalen CSV-Dateien aus `training/evaluation_results/raw`.

Wichtig:
- Der Archivordner wird nicht geladen.
- Dadurch werden alte 200-Episoden-Runs nicht mit den finalen 50-Episoden-Runs vermischt.

In [ ]:
def load_all_final_evaluation_csvs() -> pd.DataFrame:
    csv_files = sorted(EVALUATION_RAW_DIR.glob("*.csv"))

    frames = []

    for path in csv_files:
        try:
            df = pd.read_csv(path)
            df["source_file"] = show_project_path(path)
            frames.append(df)
        except Exception as exc:
            print(f"Konnte Datei nicht laden: {show_project_path(path)} | {exc}")

    if len(frames) == 0:
        print("Keine finalen Evaluation-CSV-Dateien gefunden.")
        return pd.DataFrame()

    combined_df = pd.concat(frames, ignore_index=True, sort=False)

    output_path = EVALUATION_TABLE_DIR / "all_final_evaluations_combined.csv"
    combined_df.to_csv(output_path, index=False, encoding="utf-8")

    print("Kombinierte finale Evaluation gespeichert:")
    print(show_project_path(output_path))

    return combined_df


all_eval_df = load_all_final_evaluation_csvs()
display(all_eval_df.head())

---

### 17. Evaluationsergebnisse zusammenfassen

Diese Zelle berechnet die wichtigsten Kennzahlen pro Run:

- Anzahl Episoden
- Mean Reward
- Mean Steps
- Goal Rate
- Timeout Rate
- Wall-like Failure Rate

Positive Rewards werden als Goal-Episoden interpretiert.  
Rewards kleiner oder gleich -5 werden als wall-like failures interpretiert.

In [ ]:
def summarize_evaluation_results(eval_df: pd.DataFrame) -> pd.DataFrame:
    if eval_df.empty:
        return pd.DataFrame()

    required_cols = [
        "algorithm",
        "analysis_run_id",
        "seed",
        "eval_environment",
        "episode_reward",
        "episode_steps",
    ]

    missing_cols = [col for col in required_cols if col not in eval_df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"Fehlende Spalten in eval_df: {missing_cols}")

    group_cols = ["algorithm", "analysis_run_id", "seed", "eval_environment"]

    if "eval_mode" in eval_df.columns:
        group_cols.append("eval_mode")
    elif "deterministic" in eval_df.columns:
        group_cols.append("deterministic")

    summary_df = (
        eval_df
        .groupby(group_cols)
        .agg(
            episodes=("episode", "count"),
            mean_reward=("episode_reward", "mean"),
            std_reward=("episode_reward", "std"),
            mean_steps=("episode_steps", "mean"),
            std_steps=("episode_steps", "std"),
            goals=("episode_reward", lambda x: (x > 0).sum()),
            timeouts=("episode_steps", lambda x: (x == EVAL_MAX_STEPS_PER_EPISODE).sum()),
            wall_like_failures=("episode_reward", lambda x: (x <= -5).sum()),
        )
        .reset_index()
    )

    summary_df["goal_rate"] = summary_df["goals"] / summary_df["episodes"]
    summary_df["timeout_rate"] = summary_df["timeouts"] / summary_df["episodes"]
    summary_df["wall_like_rate"] = summary_df["wall_like_failures"] / summary_df["episodes"]

    output_path = EVALUATION_TABLE_DIR / "evaluation_summary.csv"
    summary_df.to_csv(output_path, index=False, encoding="utf-8")

    print("Evaluation Summary gespeichert:")
    print(show_project_path(output_path))

    return summary_df


evaluation_summary_df = summarize_evaluation_results(all_eval_df)
display(evaluation_summary_df)

---

### 18. Beste Modelle für Transfer auswählen

Auf Basis der Env1-Evaluation werden die besten Modelle für die Transfer-Evaluation ausgewählt.

Kategorien:

- best_A2C_deterministic
- best_A2C_stochastic
- best_DQN_deterministic
- best_DQN_stochastic
- best_PPO

Auswahlkriterien in Reihenfolge:

1. höchste Goal Rate
2. höchster Mean Reward
3. niedrigste Timeout Rate
4. niedrigste Wall-like Failure Rate
5. niedrigste Mean Episode Length

Oder einfach gesagt: Goal Rate > Mean Reward > Timeout Rate > Wall-like Failure Rate > Mean Episode Length

Wenn ein Algorithmus keine Goals erreicht, kann trotzdem das relativ beste Modell anhand der übrigen Kriterien ausgewählt werden.

In [ ]:
def select_best_model(
    summary_df: pd.DataFrame,
    algorithm: str,
    eval_mode: str | None = None,
    eval_environment: str = "Env1",
) -> pd.Series:
    df = summary_df[
        (summary_df["algorithm"] == algorithm) &
        (summary_df["eval_environment"] == eval_environment)
    ].copy()

    if eval_mode is not None and "eval_mode" in df.columns:
        df = df[df["eval_mode"] == eval_mode]

    if df.empty:
        raise ValueError(
            f"Keine Ergebnisse gefunden für algorithm={algorithm}, "
            f"eval_mode={eval_mode}, eval_environment={eval_environment}"
        )

    df = df.sort_values(
        by=["goal_rate", "mean_reward", "timeout_rate", "wall_like_rate", "mean_steps"],
        ascending=[False, False, True, True, True],
    )

    return df.iloc[0]


best_models = {}

for algorithm, eval_mode, label in [
    ("A2C", "deterministic", "best_A2C_deterministic"),
    ("A2C", "stochastic", "best_A2C_stochastic"),
    ("DQN", "deterministic", "best_DQN_deterministic"),
    ("DQN", "stochastic", "best_DQN_stochastic"),
]:
    try:
        best_models[label] = select_best_model(
            evaluation_summary_df,
            algorithm=algorithm,
            eval_mode=eval_mode,
            eval_environment="Env1",
        )
    except Exception as exc:
        print(f"{label} konnte nicht ausgewählt werden:", exc)

try:
    best_models["best_PPO"] = select_best_model(
        evaluation_summary_df,
        algorithm="PPO",
        eval_mode=None,
        eval_environment="Env1",
    )
except Exception as exc:
    print("best_PPO konnte nicht ausgewählt werden:", exc)

best_models_df = (
    pd.DataFrame(best_models)
    .T
    .reset_index()
    .rename(columns={"index": "best_model_label"})
)

best_models_path = EVALUATION_TABLE_DIR / "best_models_for_transfer.csv"
best_models_df.to_csv(best_models_path, index=False, encoding="utf-8")

print("Best Models gespeichert:")
print(show_project_path(best_models_path))

display(best_models_df)

---

### 19. Transfer-Evaluation vorbereiten

Die Transfer-Evaluation wird nur mit den besten Modellen aus Env1 durchgeführt.

Geplant:

- best_A2C_deterministic → Env2 und Env3
- best_A2C_stochastic → Env2 und Env3
- best_DQN_deterministic → Env2 und Env3
- best_DQN_stochastic → Env2 und Env3
- best_PPO → Env2 und Env3 über Unity Inference

Damit wird nicht jede Seed-Kombination übertragen, sondern nur die bestperformenden Modelle je Kategorie.

In [ ]:
def evaluate_best_sb3_model(
    best_model_label: str,
    eval_environment: str,
) -> pd.DataFrame:
    if best_model_label not in best_models:
        raise ValueError(f"{best_model_label} nicht in best_models gefunden.")

    row = best_models[best_model_label]

    algorithm = row["algorithm"]
    seed = int(row["seed"])

    deterministic = True

    if "eval_mode" in row.index:
        deterministic = row["eval_mode"] == "deterministic"

    print("Best Model:", best_model_label)
    print("Algorithm:", algorithm)
    print("Seed:", seed)
    print("Deterministic:", deterministic)
    print("Target Environment:", eval_environment)

    return evaluate_selected_sb3_model(
        algorithm=algorithm,
        seed=seed,
        eval_environment=eval_environment,
        deterministic=deterministic,
    )

---

### 20. SB3 Transfer-Runs

Diese Zellen führen die Transfer-Evaluation für die besten A2C- und DQN-Modelle aus.

Vor jedem Run:
- passende Unity-Szene öffnen
- Env2 oder Env3 auswählen
- EpisodeLogger Run ID passend setzen
- Max Episodes: 50
- Python-Zelle starten
- Unity Play drücken

#### 20.1 Transfer best A2C deterministic to Env2

In [ ]:
transfer_best_a2c_deterministic_env2 = evaluate_best_sb3_model(
    best_model_label="best_A2C_deterministic",
    eval_environment="Env2",
)

#### 20.2 Transfer best A2C deterministic to Env3

In [ ]:
transfer_best_a2c_deterministic_env3 = evaluate_best_sb3_model(
    best_model_label="best_A2C_deterministic",
    eval_environment="Env3",
)

#### 20.3 Transfer best A2C stochastic to Env2

In [ ]:
transfer_best_a2c_stochastic_env2 = evaluate_best_sb3_model(
    best_model_label="best_A2C_stochastic",
    eval_environment="Env2",
)

#### 20.4 Transfer best A2C stochastic to Env3

In [ ]:
transfer_best_a2c_stochastic_env3 = evaluate_best_sb3_model(
    best_model_label="best_A2C_stochastic",
    eval_environment="Env3",
)

#### 20.5 Transfer best DQN deterministic to Env2

In [ ]:
transfer_best_dqn_deterministic_env2 = evaluate_best_sb3_model(
    best_model_label="best_DQN_deterministic",
    eval_environment="Env2",
)

#### 20.6 Transfer best DQN deterministic to Env3

In [ ]:
transfer_best_dqn_deterministic_env3 = evaluate_best_sb3_model(
    best_model_label="best_DQN_deterministic",
    eval_environment="Env3",
)

#### 20.7 Transfer best DQN stochastic to Env2

In [ ]:
transfer_best_dqn_stochastic_env2 = evaluate_best_sb3_model(
    best_model_label="best_DQN_stochastic",
    eval_environment="Env2",
)

#### 20.8 Transfer best DQN stochastic to Env3

In [ ]:
transfer_best_dqn_stochastic_env3 = evaluate_best_sb3_model(
    best_model_label="best_DQN_stochastic",
    eval_environment="Env3",
)

---

### 21. PPO Transfer über Unity Inference

PPO-Transfer wird separat in Unity durchgeführt.

Vorgehen:

1. Bestes PPO-Modell aus Env1 anhand der Summary bestimmen.
2. Passendes ONNX-Modell in Unity beim MazeAgent eintragen.
3. Behavior Type auf `Inference Only` setzen.
4. Env2 oder Env3 öffnen.
5. EpisodeLogger auf 50 Episoden setzen.
6. Run ID passend setzen, z. B.:
   - best_PPO_eval_Env2
   - best_PPO_eval_Env3

Die PPO-Ergebnisse werden danach ebenfalls als CSV in die finale Auswertung aufgenommen.

In [ ]:
ppo_transfer_plan_df = pd.DataFrame([
    {
        "best_model_label": "best_PPO",
        "eval_environment": "Env2",
        "episodes": EVAL_EPISODES,
        "run_id": "best_PPO_eval_Env2",
    },
    {
        "best_model_label": "best_PPO",
        "eval_environment": "Env3",
        "episodes": EVAL_EPISODES,
        "run_id": "best_PPO_eval_Env3",
    },
])

display(ppo_transfer_plan_df)

---

### 22. Abschluss der Evaluation

Nach allen finalen Evaluationen werden die CSV-Dateien erneut geladen und die Summary wird aktualisiert.

Wichtig:
- Archivierte 200-Episoden-Runs werden nicht geladen.
- Finale Vergleichsbasis sind 50 Episoden pro Run.
- A2C und DQN werden nach deterministic/stochastic getrennt ausgewertet.
- PPO wird separat als Unity-ML-Agents-Inference-Ergebnis dokumentiert.

In [ ]:
all_eval_df = load_all_final_evaluation_csvs()
evaluation_summary_df = summarize_evaluation_results(all_eval_df)

display(evaluation_summary_df)

---

### 23. Nächster Schritt

Die Visualisierungen werden im finalen Ergebnisnotebook erstellt.

Geplante Plots:

1. Goal Rate nach Algorithmus, Seed und Evaluationsmodus in Env1
2. Mean Reward nach Algorithmus, Seed und Evaluationsmodus in Env1
3. Timeout Rate nach Algorithmus, Seed und Evaluationsmodus in Env1
4. Wall-like Failure Rate nach Algorithmus, Seed und Evaluationsmodus in Env1
5. Deterministic vs. stochastic Vergleich für A2C und DQN
6. Transfervergleich der Best Models auf Env1, Env2 und Env3
7. Random Baseline vs. trainierte Modelle